# Load Llama 3.1 Instruct (8B)

This notebook loads `meta-llama/Llama-3.1-8B-Instruct` into memory with Hugging Face Transformers.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc

In [3]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device)

W1105 13:10:39.924000 57155 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
messages = [
    {"role": "user", "content": "Write one short sentence about learning NLP."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    output_tokens[0, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(response.strip())

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Learning NLP (Natural Language Processing) involves understanding how to analyze, interpret, and generate human language using computational methods to develop intelligent systems that can comprehend and interact with humans more effectively.


In [5]:
def steer_hook(module, inputs, output):
    hidden = output.clone()
    hidden[:, :-1, :] += torch.ones((1, 4096)).to(device)
    return hidden

handle = model.model.layers[4].register_forward_hook(steer_hook)

messages = [
    {"role": "user", "content": "Write one short sentence about learning NLP."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    output_tokens[0, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(response.strip())

handle.remove()

Diazneseonl is not just the act of is the act of is the act is the act of is the act of is the act is the act of
 act is the act of


In [6]:
# Keep a reference to one loaded weight tensor for inspection.
weight_name, weights = next(model.named_parameters())
print(f"{weight_name}: shape={tuple(weights.shape)}, device={weights.device}, dtype={weights.dtype}")

model.embed_tokens.weight: shape=(128256, 4096), device=mps:0, dtype=torch.float16


In [7]:
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()